# Formação de Pares — Cointegração Não Paramétrica (kernel local linear)

Terceiro método do TCC. A relação de equilíbrio deixa de ser uma reta e passa a ser uma
curva desconhecida, estimada por **regressão local linear**:

$$Q_{\text{dep}}(t) = g\big(Q_{\text{indep}}(t)\big) + u_t$$

Em cada ponto $x$, o local linear resolve

$$\big(\hat a(x), \hat b(x)\big) = \arg\min_{a,b} \sum_s \big[Q_{\text{dep}}(s) - a - b\,(Q_{\text{indep}}(s) - x)\big]^2 \, K\!\left(\frac{Q_{\text{indep}}(s) - x}{h}\right)$$

e devolve a **curva de equilíbrio** $\hat g(x) = \hat a(x)$ e o **hedge ratio local**
$\hat\beta(x) = \hat b(x) \approx \hat g'(x)$.

### O que este notebook faz

Para cada par candidato de cada janela de formação:

1. Normaliza os preços pelo início da formação, $Q_i(t) = P_i(t)/P_i(0)$ — a mesma base dos
   outros dois métodos.
2. Escolhe a **bandwidth** $h$ por validação cruzada **unilateral** (origem móvel): cada dia é
   previsto só com o passado de pelo menos 20 pregões de idade (seção 3).
3. Calcula o erro de previsão um passo à frente (só passado) no $h$ escolhido → $\mu$ e $\sigma$ do spread, que
   padronizam o z-score na negociação.
4. Registra diagnósticos: ADF dos resíduos, dispersão do $\hat\beta(x)$ local e o $h$ que o
   LS-CV *leave-one-out* teria escolhido.

**A curva em si não é salva.** Na negociação ela é re-estimada todo dia com os dados até
$t-1$ (formação + trecho decorrido da negociação) — ver `pairs_trading_np.ipynb`. O que sai
daqui é o $h$, que fica fixo no período de negociação.

### Origem dos pares — provisória

Por enquanto os candidatos são **os pares da cointegração linear** (`ORIGEM_PARES`), com a
direção do OLS que ela escolheu (`direcao`). Isso torna a comparação pareada: contra o
estático e o dinâmico, a única diferença é reta × curva. A seleção própria do método não
paramétrico deve entrar **neste notebook** quando for implementada.

**Base teórica:** slides *Cointegração Não Paramétrica via Kernel — Aplicação em Pairs Trading*
(ECO 02091, FCE/UFRGS), `articles/`. Wang & Phillips (2009a, 2009b, 2016);
Liang, Lin & Hsiao (2015).

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

from kernel_np import loclin, cv_bloco, cv_unilateral, residuos_loo, residuos_unilaterais

warnings.filterwarnings('ignore')

# ── Origem dos pares candidatos ───────────────────────────────────────────────
# PROVISÓRIO: pares da cointegração linear (mesmos pares → comparação pareada).
# Precisa ter as colunas da formação linear: janela, datas, ativo1, ativo2, direcao.
ORIGEM_PARES = '../cointegration/output/pares/pares_formados_coint.csv'

# ── Bandwidth ─────────────────────────────────────────────────────────────────
# Grade em unidades do desvio-padrão de Q_indep na formação, para que o mesmo h_rel
# signifique a mesma suavização em pares de volatilidade diferente.
# np.inf = peso igual para todas as observações = OLS: a reta é o caso-limite e
# entra explicitamente na disputa.
GRADE_H_REL = np.r_[np.geomspace(0.05, 5, 25), np.inf]

# Critério de validação cruzada (ver seção 3):
#   'bloco'      → bilateral: prevê t sem os pregões |s − t| ≤ L_BLOCO (usa passado E futuro)
#   'unilateral' → origem móvel: prevê t só com o passado s ≤ t − GAP_UNILATERAL,
#                  avaliando t = T_MIN_UNILATERAL … fim da formação (como na negociação)
# PADRÃO desde 2026-10-19 (decisão do usuário): unilateral, gap 20, σ unilateral.
# A CV de bloco usa dados posteriores ao dia previsto, o que não existe na negociação.
CRITERIO_H = 'unilateral'

# Unilateral: distância mínima entre o passado usado e o dia previsto.
#   1  → igual ao monitoramento da negociação (curva até t−1)
#   20 → só passagens do preço com pelo menos ~1 mês de idade
GAP_UNILATERAL   = 20
T_MIN_UNILATERAL = 126    # primeiro dia avaliado (~6 meses de passado acumulado)

# μ e σ do spread, que padronizam o z na negociação:
#   'loo'        → resíduo leave-one-out (usa vizinhos dos dois lados no tempo)
#   'unilateral' → resíduo de previsão um passo à frente (s ≤ t−1, t ≥ T_MIN_UNILATERAL):
#                  a mesma natureza do spread monitorado na negociação
SIGMA_MODO = 'unilateral'

# Tamanho do bloco excluído na CV: ĝ(x_t) é estimado sem os pregões a menos de
# L_BLOCO dias de t. Com o AR(1) mediano do resíduo linear (≈0,88, meia-vida ≈5 pregões),
# 20 pregões ≈ 4 meias-vidas → correlação residual 0,88^20 ≈ 0,08. Ver seção 3.
L_BLOCO = 20

# Diagnóstico: também calcula o h do LS-CV leave-one-out (Liang, Lin & Hsiao, 2015)
# para comparação. Dobra o tempo de execução.
CALCULAR_H_LOO = True

# Arquivo de saída — rodadas de sensibilidade (outro L_BLOCO, outra grade) usam outro nome
ARQUIVO_SAIDA = 'output/pares/pares_formados_np.csv'

META_COLS = ['year', 'semester', 'period', 'day_in_semester', 'total_days_sem']

## 1. Carregando os dados

In [ ]:
df = pd.read_csv('../../data_bases/base_unificada.csv', parse_dates=['date'])
df = df.set_index('date')
meses_por_linha = pd.PeriodIndex(df.index, freq='M')

pares_origem = pd.read_csv(ORIGEM_PARES)

print(f"Base de preços   : {df.shape[0]:,} dias × {len([c for c in df.columns if c not in META_COLS]):,} tickers")
print(f"Pares candidatos : {len(pares_origem):,} ({pares_origem['janela'].nunique()} janelas)")
print(f"Origem           : {ORIGEM_PARES}")

## 2. Direção da regressão

Como a curva não paramétrica **não é simétrica** — $g$ de $Q_2$ em $Q_1$ não é a inversa de
$g$ de $Q_1$ em $Q_2$ —, é preciso fixar qual ativo é a variável dependente. Por enquanto
herdamos a direção que a formação linear escolheu (`direcao`):

- `'B~A'` → dependente = `ativo2`, independente = `ativo1`
- `'A~B'` → dependente = `ativo1`, independente = `ativo2`

## 3. Escolha da bandwidth

O $h$ controla o tamanho da vizinhança: pequeno → curva flexível, variância alta, risco de
sobreajuste; grande → curva suave, viés, pode esconder a não linearidade. No limite
$h \to \infty$ o local linear **é** o OLS.

### Por que não o LS-CV *leave-one-out*

O critério clássico (Liang, Lin & Hsiao, 2015) escolhe $h$ minimizando
$\sum_t [Q_{\text{dep}}(t) - \hat g_{-t}(Q_{\text{indep}}(t))]^2$, com $\hat g_{-t}$ estimado sem
a observação $t$. Com regressor I(1) isso subsuaviza: os vizinhos de $Q_{\text{indep}}(t)$ **no
eixo do preço** são quase sempre os dias $t \pm 1, t \pm 2, \dots$ — o preço passeia, então
quem está perto em nível está perto no tempo. E o resíduo é persistente (AR(1) ≈ 0,88 nos
pares lineares). Tirar só o ponto $t$ deixa os vizinhos temporais, cujo erro é correlacionado
com o de $t$, "colarem" a curva no dado. O CV então recompensa $h$ pequeno.

### Leave-block-out

Excluímos o bloco $|s - t| \le L$ (Burman, Chow & Nolan, 1994). Usamos $L = 20$ pregões: o
AR(1) mediano do resíduo linear é ≈ 0,88 (meia-vida ≈ 5 pregões), então a 20 pregões de
distância a correlação residual já caiu para $0{,}88^{20} pprox 0{,}08$.

$$\text{CV}(h) = \frac{1}{T}\sum_t \Big[Q_{\text{dep}}(t) - \hat g_{-[t-L,\,t+L]}\big(Q_{\text{indep}}(t)\big)\Big]^2$$

Com o bloco fora, $\hat g$ em $Q_{\text{indep}}(t)$ só pode vir de **outras visitas** do preço
àquele nível — que é exatamente a informação que identifica uma relação de equilíbrio. Se o
preço nunca voltou ao nível, a curva local não tem como ser validada ali, e o CV naturalmente
empurra para um $h$ maior (no limite, a reta).

### Critério adotado: CV unilateral (origem móvel)

A CV de bloco é **bilateral**: para prever o dia $t$ ela usa o passado antes do bloco e o
futuro depois dele. Na negociação isso não existe — no dia $t$ a curva só conhece o passado.
A CV unilateral (`CRITERIO_H = 'unilateral'`) reproduz essa situação dentro da formação:

$$\text{CV}_{\text{uni}}(h) = \frac{1}{T - t_{\min}}\sum_{t = t_{\min}}^{T-1} \Big[Q_{\text{dep}}(t) - \hat g_{\{s \,\le\, t - g\}}\big(Q_{\text{indep}}(t)\big)\Big]^2$$

- $g$ (`GAP_UNILATERAL`) é a idade mínima do passado usado. Com $g = 1$ a CV mede
  exatamente o erro do monitoramento; com $g = 20$ exige passagens antigas do preço, que é
  a ideia de equilíbrio do bloco olhando só para trás.
- $t_{\min}$ (`T_MIN_UNILATERAL`): os primeiros dias da formação não têm passado suficiente
  para estimar a curva e não são avaliados.

### Por que escolher uma vez só

A teoria de cointegração não paramétrica (Wang & Phillips) mostra que, com regressor I(1),
o número efetivo de observações perto de um ponto — o *tempo de permanência* (local time) do
preço naquele nível — cresce como $\sqrt{T}$, não como $T$. Balanceando viés² $O(h^4)$ contra
variância $O(1/(\sqrt{T}h))$, o $h$ ótimo decresce como $T^{-1/10}$ (derivação heurística nossa).
Wang & Phillips (2023, *Econometric Theory* 39(6)) mostram que o $h$ ótimo é na verdade uma
sequência **aleatória**, que depende do tempo de permanência do processo — o que reforça o
veto por suporte da negociação, que mede justamente essa massa local. Entre $T \approx 252$ (fim da formação) e
$T \approx 378$ (fim da negociação) isso muda o $h$ em ~4%. Por isso o $h$ é escolhido aqui e
fica fixo durante a negociação.

## 4. Funções

In [ ]:
def ajustar_par(precos_form, dep, indep):
    """
    Ajusta a cointegração não paramétrica de um par na janela de formação.

    precos_form : DataFrame com os preços da formação (colunas dep e indep), sem NaN.
    Retorna dict com h escolhido, μ e σ do spread (resíduo leave-one-out) e diagnósticos.
    """
    P0 = precos_form.iloc[0]
    y  = (precos_form[dep]   / P0[dep]).values
    x  = (precos_form[indep] / P0[indep]).values
    sx = x.std()

    grade_h = GRADE_H_REL * sx
    if CRITERIO_H == 'bloco':
        h, mse = cv_bloco(x, y, grade_h, L_BLOCO)
    elif CRITERIO_H == 'unilateral':
        h, mse = cv_unilateral(x, y, grade_h, GAP_UNILATERAL, T_MIN_UNILATERAL)
    else:
        raise ValueError(f"CRITERIO_H desconhecido: {CRITERIO_H!r}")

    if SIGMA_MODO == 'loo':
        u = residuos_loo(x, y, h)
    elif SIGMA_MODO == 'unilateral':
        u = residuos_unilaterais(x, y, h, 1, T_MIN_UNILATERAL)
    else:
        raise ValueError(f"SIGMA_MODO desconhecido: {SIGMA_MODO!r}")
    # Com a CV unilateral, o h escolhido só garante curva definida em t ≥ T_MIN olhando
    # para trás; um dia isolado no início da formação pode ficar sem vizinhos no LOO.
    n_nan = int(np.sum(~np.isfinite(u)))
    u = u[np.isfinite(u)]

    # β local nos pontos da amostra: o quanto o hedge ratio varia ao longo da curva
    _, b_loc, _ = loclin(x, x, y, h)
    beta_lin = np.polyfit(x, y, 1)[0]

    res = {
        'criterio_h':    CRITERIO_H if CRITERIO_H == 'bloco' else f'unilateral_g{GAP_UNILATERAL}',
        'sigma_modo':    SIGMA_MODO,
        'n_resid_nan':   n_nan,
        'h':             h,
        'h_rel':         h / sx,
        'h_linear':      bool(np.isinf(h)),
        'cv_mse':        np.nanmin(mse),
        'cv_mse_linear': mse[-1],                 # GRADE_H_REL termina em inf
        'spread_media':  float(np.mean(u)),
        'spread_desvio': float(np.std(u)),
        'adf_resid':     adfuller(u, autolag='AIC', regression='n')[0],
        'beta_lin':      beta_lin,
        'beta_loc_p10':  np.nanpercentile(b_loc, 10),
        'beta_loc_med':  np.nanmedian(b_loc),
        'beta_loc_p90':  np.nanpercentile(b_loc, 90),
    }
    if CALCULAR_H_LOO:
        h_loo, _ = cv_bloco(x, y, grade_h, 0)
        res['h_rel_loo'] = h_loo / sx
    return res

## 5. Processando todos os pares

Com `SIGMA_MODO = 'unilateral'` (padrão), $\mu$ e $\sigma$ vêm do erro de previsão **um passo à
frente** dentro da formação ($t \ge$ `T_MIN_UNILATERAL`, curva estimada só com $s \le t-1$): a
mesma natureza do spread monitorado na negociação. Com `SIGMA_MODO = 'loo'` vêm dos resíduos
**leave-one-out**. Em qualquer caso, não usamos os
resíduos dentro da amostra: com $h$ pequeno a curva passa perto de cada ponto e o resíduo
dentro da amostra subestima a dispersão, inflando o z-score na negociação. Com $h = \infty$ os
dois praticamente coincidem com o resíduo do OLS.

In [ ]:
COLS_ORIGEM = ['janela', 'formacao_inicio', 'formacao_fim', 'negociacao_inicio',
               'negociacao_fim', 'rank', 'ativo1', 'ativo2', 'ssd', 'direcao']

resultados = []
n_sem_dados = 0
t0 = time.time()

for k, (_, par) in enumerate(pares_origem.iterrows()):
    a1, a2 = par['ativo1'], par['ativo2']
    dep, indep = (a1, a2) if par['direcao'] == 'A~B' else (a2, a1)

    form_inicio = pd.Period(par['formacao_inicio'], freq='M')
    form_fim    = pd.Period(par['formacao_fim'],    freq='M')
    mask_form   = (meses_por_linha >= form_inicio) & (meses_por_linha <= form_fim)
    precos_form = df.loc[mask_form, [dep, indep]].dropna()

    if len(precos_form) < 2 * L_BLOCO + 10:
        n_sem_dados += 1
        continue

    resultados.append({**{c: par[c] for c in COLS_ORIGEM},
                       **ajustar_par(precos_form, dep, indep),
                       'origem': ORIGEM_PARES.split('/')[-1]})

    if (k + 1) % 1000 == 0:
        print(f"  {k + 1:6,} / {len(pares_origem):,} pares  ({time.time() - t0:5.0f} s)")

pares_np = pd.DataFrame(resultados)
print(f"\nPares ajustados : {len(pares_np):,}  | sem dados suficientes: {n_sem_dados}")
print(f"Tempo           : {time.time() - t0:.0f} s")

## 6. Resultados — quanto de não linearidade os dados sustentam?

In [ ]:
q = [0.10, 0.25, 0.50, 0.75, 0.90]

crit = (f"CV de bloco, L = {L_BLOCO} pregões" if CRITERIO_H == 'bloco' else
        f"CV unilateral, gap = {GAP_UNILATERAL}, t_min = {T_MIN_UNILATERAL}")
print(f"Bandwidth escolhida ({crit}) | μ, σ: resíduo {SIGMA_MODO}")
print(f"  pares em que a reta venceu (h = ∞) : {pares_np['h_linear'].mean():.1%}")
print(f"  h/sd(x) dos demais — quantis       : "
      + ' | '.join(f"{v:.2f}" for v in pares_np.loc[~pares_np['h_linear'], 'h_rel'].quantile(q)))
if CALCULAR_H_LOO:
    print(f"\nLS-CV leave-one-out (só diagnóstico)")
    print(f"  pares em que a reta venceu          : {np.isinf(pares_np['h_rel_loo']).mean():.1%}")
    print(f"  h/sd(x) — quantis                   : "
          + ' | '.join(f"{v:.2f}" for v in pares_np['h_rel_loo'].replace(np.inf, np.nan).quantile(q)))

ganho = 1 - pares_np['cv_mse'] / pares_np['cv_mse_linear']
print(f"\nRedução do erro de CV contra a reta (pares não lineares): mediana "
      f"{ganho[~pares_np['h_linear']].median():.1%}")

amplitude = pares_np['beta_loc_p90'] - pares_np['beta_loc_p10']
print(f"Amplitude p90−p10 do β local (pares não lineares)      : mediana "
      f"{amplitude[~pares_np['h_linear']].median():.3f}")

por_ano = (pares_np.assign(ano=pares_np['formacao_inicio'].str[:4])
                   .groupby('ano')['h_linear'].mean())
print(f"\nFração linear por ano de formação: mín {por_ano.min():.0%} | máx {por_ano.max():.0%}")

In [ ]:
pares_np.to_csv(ARQUIVO_SAIDA, index=False)
print(f"Pares salvos em: {ARQUIVO_SAIDA}")

## 7. Visualização

À esquerda, a distribuição do $h$ escolhido pelos dois critérios. À direita, o par em que a
curva mais reduziu o erro de CV em relação à reta: nuvem $(Q_{\text{indep}}, Q_{\text{dep}})$ da
formação, reta OLS, curva local linear e o hedge ratio local $\hat\beta(x)$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# ── Distribuição de h ─────────────────────────────────────────────────────────
ax = axes[0]
bins = np.geomspace(0.04, 6, 30)
ax.hist(pares_np.loc[~pares_np['h_linear'], 'h_rel'], bins=bins, alpha=0.7,
        label=(f'CV de bloco (L={L_BLOCO})' if CRITERIO_H == 'bloco' else f'CV unilateral (gap {GAP_UNILATERAL})') + ' — finito')
if CALCULAR_H_LOO:
    ax.hist(pares_np['h_rel_loo'].replace(np.inf, np.nan).dropna(), bins=bins, alpha=0.5,
            label='LS-CV leave-one-out')
ax.set_xscale('log')
ax.set_xlabel('h / sd(Q_indep)')
ax.set_ylabel('pares')
ax.set_title(f"h escolhido — reta venceu em {pares_np['h_linear'].mean():.0%} ({CRITERIO_H})", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Par mais não linear ───────────────────────────────────────────────────────
ex   = pares_np.loc[(1 - pares_np['cv_mse'] / pares_np['cv_mse_linear']).idxmax()]
dep, indep = (ex['ativo1'], ex['ativo2']) if ex['direcao'] == 'A~B' else (ex['ativo2'], ex['ativo1'])
mask = ((meses_por_linha >= pd.Period(ex['formacao_inicio'], freq='M')) &
        (meses_por_linha <= pd.Period(ex['formacao_fim'], freq='M')))
pf   = df.loc[mask, [dep, indep]].dropna()
y, x = (pf[dep] / pf[dep].iloc[0]).values, (pf[indep] / pf[indep].iloc[0]).values
xg   = np.linspace(x.min(), x.max(), 200)
g, b, _ = loclin(xg, x, y, ex['h'])
B    = np.polyfit(x, y, 1)

ax = axes[1]
ax.scatter(x, y, s=6, alpha=0.4, color='gray', label='formação')
ax.plot(xg, np.polyval(B, xg), color='#1565C0', lw=1.5, label=f'OLS (β={B[0]:.2f})')
ax.plot(xg, g, color='#E65100', lw=2, label=f"local linear (h/sd={ex['h_rel']:.2f})")
ax.set_xlabel(f'Q {indep}')
ax.set_ylabel(f'Q {dep}')
ax.set_title(f"{ex['janela']}: {dep} ~ g({indep})", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(xg, b, color='#E65100', lw=2, label='β(x) local')
ax.axhline(B[0], color='#1565C0', lw=1.2, ls='--', label='β OLS')
ax.set_xlabel(f'Q {indep}')
ax.set_ylabel('hedge ratio')
ax.set_title('Inclinação local = hedge ratio na entrada', fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
sufixo = ARQUIVO_SAIDA.split('/')[-1].replace('pares_formados_np', '').replace('.csv', '')
plt.savefig(f'output/imagens/bandwidth_e_curva_exemplo{sufixo}.png', dpi=150, bbox_inches='tight')
plt.show()